In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Convert timestamps to timezone-aware datetimes (UTC->Asia/Shanghai)
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
log_df['ts'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
error_df['ts'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Summaries for metric.csv
metric_total = metric_df.shape[0]
metric_min_ts = metric_df['ts'].min()
metric_max_ts = metric_df['ts'].max()
metric_top_pairs = (
    metric_df.groupby(['cmdb_id', 'kpi_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

# Summaries for trace.csv
trace_total = trace_df.shape[0]
trace_min_ts = trace_df['ts'].min()
trace_max_ts = trace_df['ts'].max()
trace_top_pairs = (
    trace_df.groupby(['cmdb_id', 'trace_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

# Summaries for log.csv
log_total = log_df.shape[0]
log_min_ts = log_df['ts'].min()
log_max_ts = log_df['ts'].max()
log_top_pairs = (
    log_df.groupby(['cmdb_id', 'log_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

# Summaries for error_logs.csv
error_total = error_df.shape[0]
error_min_ts = error_df['ts'].min() if error_total>0 else None
error_max_ts = error_df['ts'].max() if error_total>0 else None
error_counts_by_cmdb = (
    error_df.groupby('cmdb_id')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)
error_sample_earliest = error_df.sort_values('ts').loc[:, ['ts', 'cmdb_id', 'message']].head(5)

# Assemble compact structured summary
summary = {
    'metric': {
        'total_rows': int(metric_total),
        'min_ts': str(metric_min_ts),
        'max_ts': str(metric_max_ts),
        'top_cmdb_kpi_pairs': metric_top_pairs.reset_index(drop=True)
    },
    'trace': {
        'total_rows': int(trace_total),
        'min_ts': str(trace_min_ts),
        'max_ts': str(trace_max_ts),
        'top_cmdb_trace_pairs': trace_top_pairs.reset_index(drop=True)
    },
    'log': {
        'total_rows': int(log_total),
        'min_ts': str(log_min_ts),
        'max_ts': str(log_max_ts),
        'top_cmdb_log_pairs': log_top_pairs.reset_index(drop=True)
    },
    'error_logs': {
        'total_rows': int(error_total),
        'min_ts': str(error_min_ts) if error_min_ts is not None else None,
        'max_ts': str(error_max_ts) if error_max_ts is not None else None,
        'top_counts_by_cmdb': error_counts_by_cmdb.reset_index(drop=True),
        'earliest_sample_messages_up_to_5': error_sample_earliest.reset_index(drop=True)
    }
}

summary

```
Out[1]:
```
Summary (plain English):

- metric.csv
  - Total rows: 28,314
  - Time range (UTC+8): 2021-03-10 07:00:00 to 2021-03-10 07:30:00
  - Top (cmdb_id, kpi_name) pairs (by row count, up to 20): Mostly 31 samples each. Notable entries include many Mysql01 OSLinux network/process/CPU KPIs, IG01 CPU and JVM memory KPIs, and Tomcat04 session metrics.

- trace.csv
  - Total rows: 6,798
  - Time range (UTC+8): 2021-03-10 07:00:00 to 2021-03-10 07:30:00
  - Top (cmdb_id, trace_name) pairs (up to 20): dockerB1 trace.self.* metrics (row_count, duration_p95, duration_mean) with 31 samples; IG01 and IG02 trace.self.* and trace.to_TomcatXX.* entries generally with 30 samples.

- log.csv
  - Total rows: 396
  - Time range (UTC+8): 2021-03-10 07:00:00 to 2021-03-10 07:30:00
  - Top (cmdb_id, log_name) pairs (up to 20): Tomcat02 and apache01 have log.row_count and log.error_count (31 each); Tomcat01/03/04 also show log.row_count and log.error_count; a few gc_pause metrics appear with smaller counts.

- error_logs.csv
  - Total rows: 0
  - No timestamps or messages present (no sample rows to show).

If you want, I can return these top-pair tables as compact dataframes or extract the exact top-20 rows for each file.

The original code execution output of IPython Kernel is also provided below for reference:

{'metric': {'total_rows': 28314, 'min_ts': '2021-03-10 07:00:00+08:00', 'max_ts': '2021-03-10 07:30:00+08:00', 'top_cmdb_kpi_pairs':      cmdb_id                                           kpi_name  count
0    Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31
1    Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31
2   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31
3    Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31
4       IG01                        OSLinux-CPU_CPU_CPUUserTime     31
5       IG01                             OSLinux-CPU_CPU_CPUWio     31
6       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31
7       IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31
8       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31
9   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31
10   Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31
11   Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31
12  apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31
13      IG01                        OSLinux-CPU_CPU_CPUidleutil     31
14      IG01                            OSLinux-CPU_CPU_CPULoad     31
15      IG01                         OSLinux-CPU_CPU_CPUSysTime     31
16   Mysql01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31
17   Mysql01                         OSLinux-CPU_CPU_CPUSysTime     31
18   Mysql01                        OSLinux-CPU_CPU_CPUUserTime     31
19   Mysql01                             OSLinux-CPU_CPU_CPUWio     31}, 'trace': {'total_rows': 6798, 'min_ts': '2021-03-10 07:00:00+08:00', 'max_ts': '2021-03-10 07:30:00+08:00', 'top_cmdb_trace_pairs':      cmdb_id                       trace_name  count
0   dockerB1             trace.self.row_count     31
1   dockerB1          trace.self.duration_p95     31
2   dockerB1         trace.self.duration_mean     31
3       IG01  trace.to_Tomcat01.duration_mean     30
4       IG01         trace.self.duration_mean     30
5       IG01          trace.self.duration_p95     30
6       IG01             trace.self.row_count     30
7       IG01   trace.to_Tomcat02.duration_p95     30
8       IG01      trace.to_Tomcat02.row_count     30
9       IG01  trace.to_Tomcat03.duration_mean     30
10      IG01   trace.to_Tomcat03.duration_p95     30
11      IG01      trace.to_Tomcat03.row_count     30
12      IG01  trace.to_Tomcat04.duration_mean     30
13      IG01   trace.to_Tomcat04.duration_p95     30
14      IG01      trace.to_Tomcat04.row_count     30
15      IG02         trace.self.duration_mean     30
16      IG02          trace.self.duration_p95     30
17      IG02             trace.self.row_count     30
18      IG02  trace.to_Tomcat01.duration_mean     30
19      IG02   trace.to_Tomcat01.duration_p95     30}, 'log': {'total_rows': 396, 'min_ts': '2021-03-10 07:00:00+08:00', 'max_ts': '2021-03-10 07:30:00+08:00', 'top_cmdb_log_pairs':      cmdb_id          log_name  count
0   Tomcat02     log.row_count     31
1   Tomcat02   log.error_count     31
2   apache01   log.error_count     31
3   Tomcat04     log.row_count     31
4   Tomcat03   log.error_count     31
5   apache01     log.row_count     31
6   Tomcat04   log.error_count     31
7   Tomcat03     log.row_count     31
8   Tomcat01   log.error_count     30
9   Tomcat01     log.row_count     30
10  apache02     log.row_count     27
11  apache02   log.error_count     27
12   Mysql01     log.row_count      8
13   Mysql01   log.error_count      8
14  Tomcat02  log.gc_pause_max      6
15  Tomcat01  log.gc_pause_max      4
16  Tomcat03  log.gc_pause_max      4
17  Tomcat04  log.gc_pause_max      4}, 'error_logs': {'total_rows': 0, 'min_ts': None, 'max_ts': None, 'top_counts_by_cmdb': Empty DataFrame
Columns: [cmdb_id, count]
Index: [], 'earliest_sample_messages_up_to_5': Empty DataFrame
Columns: [ts, cmdb_id, message]
Index: []}}```
```

In [ ]:
# In[2]:

import pandas as pd

# incident window in UTC+8
incident_start = pd.Timestamp('2021-03-10 07:00:00').tz_localize(tz)
incident_end = pd.Timestamp('2021-03-10 07:30:00').tz_localize(tz)

# Helper to compute full stats and incident-window stats for a dataframe
def compute_kpi_stats(df, comp_col, kpi_col):
    # full aggregated stats (use full series BEFORE filtering)
    grp = df.groupby([comp_col, kpi_col])['value']
    full_stats = grp.agg(
        count='count',
        min_value='min',
        max_value='max',
        p50=lambda x: x.quantile(0.50),
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95)
    ).reset_index()
    
    # incident-window rows
    df_inc = df[(df['ts'] >= incident_start) & (df['ts'] <= incident_end)]
    if not df_inc.empty:
        inc_grp = df_inc.groupby([comp_col, kpi_col])
        inc_stats = inc_grp.agg(
            incident_count=('value','count'),
            incident_min_ts=('ts','min'),
            incident_max_ts=('ts','max')
        ).reset_index()
    else:
        # empty incident stats
        inc_stats = pd.DataFrame(columns=[comp_col, kpi_col, 'incident_count', 'incident_min_ts', 'incident_max_ts'])
    
    # merge full and incident stats
    merged = pd.merge(full_stats, inc_stats, on=[comp_col, kpi_col], how='left')
    merged['incident_count'] = merged['incident_count'].fillna(0).astype(int)
    # format incident ts as strings, empty if missing
    merged['incident_min_ts'] = merged['incident_min_ts'].astype('datetime64[ns, {}]'.format(tz.zone)).astype(str).replace('NaT','')
    merged['incident_max_ts'] = merged['incident_max_ts'].astype('datetime64[ns, {}]'.format(tz.zone)).astype(str).replace('NaT','')
    
    # rename for uniform output
    merged = merged.rename(columns={
        comp_col: 'component',
        kpi_col: 'kpi_name',
        'min_value': 'min',
        'max_value': 'max'
    })
    
    # reorder columns
    cols = ['component','kpi_name','count','p50','p90','p95','min','max','incident_count','incident_min_ts','incident_max_ts']
    merged = merged[cols]
    
    return merged

# Compute for metric, trace, log
full_metric_kpi_stats = compute_kpi_stats(metric_df, 'cmdb_id', 'kpi_name')
full_trace_kpi_stats  = compute_kpi_stats(trace_df,  'cmdb_id', 'trace_name')
full_log_kpi_stats    = compute_kpi_stats(log_df,    'cmdb_id', 'log_name')

# Top-20 by overall count for compact display
metric_top20 = full_metric_kpi_stats.sort_values('count', ascending=False).head(20).reset_index(drop=True)
trace_top20  = full_trace_kpi_stats.sort_values('count', ascending=False).head(20).reset_index(drop=True)
log_top20    = full_log_kpi_stats.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# Display the three compact tables (top 20 each). Also keep full_* variables in kernel as requested.
metric_top20, trace_top20, log_top20

```
Out[2]:
```
# Plain English summary of the computed statistics (compact)
summary = (
    "Summary of KPI statistics and incident-window counts (incident window: 2021-03-10 07:00:00 to 2021-03-10 07:30:00 UTC+8):\n\n"
    "metric.csv (top 20 component+KPI by total rows):\n"
    "- Most top metric pairs have 31 samples each and all 31 samples fall inside the incident window (incident_count = 31).\n"
    "- Notable IG01 metrics: CPU user ~25% (p50 ≈ 25.13), CPU idle ~74% (p50 ≈ 74.37), CPULoad ≈1.0 (p50=1.0); JVM HeapMemoryMax ≈2.04e9, HeapMemoryUsed p50 ≈3.85e8.\n"
    "- Many Mysql01 OSLinux network/process/CPU KPIs appear in the top list (each with 31 samples), with min/p50/p90/p95 typically small or zero for network/process boolean-like metrics.\n\n"
    "trace.csv (top 20 component+trace by total rows):\n"
    "- dockerB1 trace.self.* metrics: 31 samples each; row_count p50=367, duration_mean p50≈0.00546s, duration_p95 p50≈0.01s — all 31 samples inside the incident window.\n"
    "- IG01 and IG02 trace.self.* and trace.to_TomcatXX.* entries mostly have 30 samples each (incident_count = 30); typical p50 duration_mean values are ~0.05–0.08s, while p95 values are higher (~0.16–0.56s depending on route).\n"
    "- For IG01/IG02 many trace row_count p50 values are ~200–210.\n\n"
    "log.csv (top 20 component+log by total rows):\n"
    "- Top log pairs (Tomcat02, Tomcat03, Tomcat04, apache01) have 31 samples and all these samples are within the incident window (incident_count = their total count).\n"
    "- Typical log.row_count p50 examples: apache01 p50=107, Tomcat04 p50=54, Tomcat02 p50=50. log.error_count for many Tomcats/apache entries is 0 (p50=0).\n"
    "- A few gc_pause_max series have fewer samples (4–6) with small pause values (p50 ≈ 0.05–0.08s).\n\n"
    "error_logs.csv:\n"
    "- No rows found; no error log samples to report.\n\n"
    "Overall: the top component-KPI series are well-covered within the incident window (most incident_count == overall count). Key resource values of interest: IG01 JVM heap and CPU stats and dockerB1 trace counts/durations — these stand out numerically and are good candidates for further root-cause inspection."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   component                                           kpi_name  count           p50           p90           p95           min           max  incident_count            incident_min_ts            incident_max_ts
0    Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
1    Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
2   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
3    Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
4       IG01                        OSLinux-CPU_CPU_CPUUserTime     31  2.513510e+01  2.516210e+01  2.516939e+01  2.506340e+01  2.517880e+01              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
5       IG01                             OSLinux-CPU_CPU_CPUWio     31  2.490000e-02  2.660350e-01  2.889500e-01  1.247200e-02  3.450000e-01              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
6       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
7       IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  3.846008e+08  7.653074e+08  8.075072e+08  4.131126e+07  8.549786e+08              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
8       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  5.792551e+07  5.792551e+07  5.792551e+07  5.792551e+07  5.792551e+07              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
9   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
10   Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31  4.000000e-01  4.000000e-01  4.000000e-01  3.000000e-01  4.000000e-01              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
11   Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31  7.000000e+00  7.000000e+00  7.000000e+00  7.000000e+00  7.000000e+00              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
12  apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  6.530341e+06  6.531063e+06  6.531155e+06  6.529439e+06  6.531245e+06              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
13      IG01                        OSLinux-CPU_CPU_CPUidleutil     31  7.436910e+01  7.456965e+01  7.457960e+01  7.397938e+01  7.459360e+01              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
14      IG01                            OSLinux-CPU_CPU_CPULoad     31  1.000000e+00  1.090000e+00  1.095000e+00  1.000000e+00  1.250000e+00              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
15      IG01                         OSLinux-CPU_CPU_CPUSysTime     31  3.243000e-01  3.658000e-01  3.887075e-01  1.913000e-01  4.115570e-01              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
16   Mysql01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  6.530289e+06  6.531012e+06  6.531102e+06  6.529386e+06  6.531192e+06              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
17   Mysql01                         OSLinux-CPU_CPU_CPUSysTime     31  3.116170e-01  4.569000e-01  4.966000e-01  2.700680e-01  5.274740e-01              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
18   Mysql01                        OSLinux-CPU_CPU_CPUUserTime     31  1.911250e-01  2.452000e-01  2.513500e-01  1.579000e-01  2.534280e-01              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
19   Mysql01                             OSLinux-CPU_CPU_CPUWio     31  3.241360e-01  8.765000e-01  1.001096e+00  9.560000e-02  1.196709e+00              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00,    component                         kpi_name  count         p50         p90         p95         min         max  incident_count            incident_min_ts            incident_max_ts
0   dockerB1             trace.self.row_count     31  367.000000  628.000000  653.000000    1.000000  757.000000              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
1   dockerB1          trace.self.duration_p95     31    0.010000    0.011000    0.011000    0.002000    0.011000              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
2   dockerB1         trace.self.duration_mean     31    0.005459    0.009441    0.010628    0.002000    0.016831              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
3       IG01  trace.to_Tomcat01.duration_mean     30    0.066180    0.124863    0.139749    0.031111    0.156269              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
4       IG01         trace.self.duration_mean     30    0.075251    0.101785    0.108692    0.034466    0.119799              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
5       IG01          trace.self.duration_p95     30    0.282425    0.554625    0.560200    0.085000    0.643000              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
6       IG01             trace.self.row_count     30  209.000000  268.200000  276.600000  136.000000  304.000000              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
7       IG01   trace.to_Tomcat02.duration_p95     30    0.184900    0.541030    0.670275    0.053250    0.811100              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
8       IG01      trace.to_Tomcat02.row_count     30   25.000000   32.100000   33.000000   14.000000   40.000000              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
9       IG01  trace.to_Tomcat03.duration_mean     30    0.058884    0.113006    0.123078    0.029143    0.180400              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
10      IG01   trace.to_Tomcat03.duration_p95     30    0.161150    0.544960    0.564630    0.056300    0.679800              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
11      IG01      trace.to_Tomcat03.row_count     30   24.000000   37.300000   41.100000   12.000000   43.000000              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
12      IG01  trace.to_Tomcat04.duration_mean     30    0.073115    0.126960    0.138023    0.030294    0.143857              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
13      IG01   trace.to_Tomcat04.duration_p95     30    0.247075    0.560150    0.614877    0.053600    0.997000              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
14      IG01      trace.to_Tomcat04.row_count     30   26.000000   34.200000   36.550000   15.000000   39.000000              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
15      IG02         trace.self.duration_mean     30    0.078537    0.117003    0.126602    0.044875    0.130214              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
16      IG02          trace.self.duration_p95     30    0.343250    0.590350    0.741167    0.097150    0.981000              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
17      IG02             trace.self.row_count     30  203.000000  248.800000  257.100000  140.000000  260.000000              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
18      IG02  trace.to_Tomcat01.duration_mean     30    0.058181    0.088502    0.139672    0.023000    0.207316              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
19      IG02   trace.to_Tomcat01.duration_p95     30    0.185650    0.453080    0.881915    0.048800    1.160400              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00,    component          kpi_name  count      p50      p90       p95    min     max  incident_count            incident_min_ts            incident_max_ts
0   Tomcat02     log.row_count     31   50.000   62.000   66.5000   1.00   76.00              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
1   Tomcat02   log.error_count     31    0.000    0.000    0.0000   0.00    0.00              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
2   apache01   log.error_count     31    0.000    0.000    0.0000   0.00    0.00              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
3   Tomcat04     log.row_count     31   54.000   66.000   69.5000   1.00   75.00              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
4   Tomcat03   log.error_count     31    0.000    0.000    0.0000   0.00    0.00              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
5   apache01     log.row_count     31  107.000  210.000  231.0000   1.00  235.00              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
6   Tomcat04   log.error_count     31    0.000    0.000    0.0000   0.00    0.00              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
7   Tomcat03     log.row_count     31   51.000   67.000   73.5000   1.00   82.00              31  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
8   Tomcat01   log.error_count     30    0.000    0.000    0.0000   0.00    0.00              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
9   Tomcat01     log.row_count     30   54.500   68.100   71.2000  31.00   77.00              30  2021-03-10 07:00:00+08:00  2021-03-10 07:29:00+08:00
10  apache02     log.row_count     27   98.000  123.400  126.1000   3.00  132.00              27  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
11  apache02   log.error_count     27    0.000    0.000    0.0000   0.00    0.00              27  2021-03-10 07:00:00+08:00  2021-03-10 07:30:00+08:00
12   Mysql01     log.row_count      8    1.000    3.000    3.0000   1.00    3.00               8  2021-03-10 07:05:00+08:00  2021-03-10 07:30:00+08:00
13   Mysql01   log.error_count      8    0.000    0.000    0.0000   0.00    0.00               8  2021-03-10 07:05:00+08:00  2021-03-10 07:30:00+08:00
14  Tomcat02  log.gc_pause_max      6    0.050    0.060    0.0600   0.03    0.06               6  2021-03-10 07:00:00+08:00  2021-03-10 07:26:00+08:00
15  Tomcat01  log.gc_pause_max      4    0.065    0.077    0.0785   0.05    0.08               4  2021-03-10 07:03:00+08:00  2021-03-10 07:27:00+08:00
16  Tomcat03  log.gc_pause_max      4    0.060    0.074    0.0770   0.03    0.08               4  2021-03-10 07:08:00+08:00  2021-03-10 07:26:00+08:00
17  Tomcat04  log.gc_pause_max      4    0.075    0.087    0.0885   0.07    0.09               4  2021-03-10 07:05:00+08:00  2021-03-10 07:29:00+08:00)```
```

In [ ]:
# In[3]:

import numpy as np

# Reuse variables: metric_df, full_metric_kpi_stats, tz, incident_start, incident_end

# Filter component+KPIs that have incident_count > 0
kpis_with_incident = full_metric_kpi_stats[full_metric_kpi_stats['incident_count'] > 0].copy()

records = []
# We'll also build a full anomalies table (per-row) for later inspection
anomaly_rows = []

for _, row in kpis_with_incident.iterrows():
    comp = row['component']
    kpi = row['kpi_name']
    overall_count = int(row['count'])
    incident_count = int(row['incident_count'])
    global_p95 = float(row['p95'])
    global_min = float(row['min'])
    global_max = float(row['max'])
    
    # incident-window rows for this comp+kpi
    inc_rows = metric_df[
        (metric_df['cmdb_id'] == comp) &
        (metric_df['kpi_name'] == kpi) &
        (metric_df['ts'] >= incident_start) &
        (metric_df['ts'] <= incident_end)
    ].copy()
    
    if inc_rows.empty:
        incident_max_value = np.nan
    else:
        incident_max_value = float(inc_rows['value'].max())
    
    # define anomaly condition per rules:
    # If p95==min==max==0, treat no-anomaly unless value > 0 (i.e., anomaly if value > 0)
    if (global_p95 == 0.0) and (global_min == 0.0) and (global_max == 0.0):
        inc_anoms = inc_rows[inc_rows['value'] > 0]
    else:
        inc_anoms = inc_rows[inc_rows['value'] >= global_p95]
    
    anomaly_count = int(inc_anoms.shape[0])
    if anomaly_count > 0:
        earliest_anomaly_ts = inc_anoms['ts'].min().tz_convert(tz)
        earliest_anomaly_ts_str = str(earliest_anomaly_ts)
    else:
        earliest_anomaly_ts_str = ''
    
    # severity ratio
    if global_p95 == 0:
        severity_ratio = None
    else:
        # handle NaN incident_max_value
        if np.isnan(incident_max_value):
            severity_ratio = None
        else:
            severity_ratio = float(incident_max_value) / float(global_p95)
    
    records.append({
        'component': comp,
        'kpi_name': kpi,
        'overall_count': overall_count,
        'incident_count': incident_count,
        'global_p95': global_p95,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_ts': earliest_anomaly_ts_str,
        'incident_max_value': incident_max_value,
        'severity_ratio': severity_ratio
    })
    
    # store per-row anomalies into anomaly_rows for full table
    if anomaly_count > 0:
        ar = inc_anoms.loc[:, ['timestamp','cmdb_id','kpi_name','value','ts']].copy()
        ar['global_p95'] = global_p95
        anomaly_rows.append(ar)

# full detected anomalies table (row-level). Concatenate if any
if anomaly_rows:
    metric_anomalies = pd.concat(anomaly_rows, ignore_index=True)
else:
    # empty table with expected columns
    metric_anomalies = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value','ts','global_p95'])

# per-component+kpi summary table
metric_anomalies_summary = pd.DataFrame.from_records(records)

# Sort by severity_ratio desc (put None/NaN last), then by anomaly_count desc
metric_anomalies_summary['severity_sort'] = metric_anomalies_summary['severity_ratio'].apply(lambda x: -np.inf if x is None else x)
# Use severity_sort for sorting descending; None become -inf so appear last after negation; then anomaly_count desc
metric_anomalies_summary_sorted = metric_anomalies_summary.sort_values(
    by=['severity_sort','anomaly_count'],
    ascending=[False, False]
).drop(columns=['severity_sort']).reset_index(drop=True)

# Top-20 by severity
metric_anomalies_top20 = metric_anomalies_summary_sorted.head(20).loc[:, [
    'component','kpi_name','overall_count','incident_count','global_p95',
    'anomaly_count','earliest_anomaly_ts','incident_max_value','severity_ratio'
]]

# If no anomalies found at P95 globally, compute relaxed using P90
metric_anomalies_relaxed = None
metric_anomalies_relaxed_top20 = pd.DataFrame()
if metric_anomalies['timestamp'].empty:
    # compute using P90
    records_relaxed = []
    anomaly_rows_relaxed = []
    for _, row in kpis_with_incident.iterrows():
        comp = row['component']
        kpi = row['kpi_name']
        overall_count = int(row['count'])
        incident_count = int(row['incident_count'])
        global_p90 = float(row['p90'])
        global_min = float(row['min'])
        global_max = float(row['max'])

        inc_rows = metric_df[
            (metric_df['cmdb_id'] == comp) &
            (metric_df['kpi_name'] == kpi) &
            (metric_df['ts'] >= incident_start) &
            (metric_df['ts'] <= incident_end)
        ].copy()
        if inc_rows.empty:
            incident_max_value = np.nan
        else:
            incident_max_value = float(inc_rows['value'].max())
        
        # anomaly condition for relaxed: similar special-case handling if p90==min==max==0 then value>0
        if (global_p90 == 0.0) and (global_min == 0.0) and (global_max == 0.0):
            inc_anoms = inc_rows[inc_rows['value'] > 0]
        else:
            inc_anoms = inc_rows[inc_rows['value'] >= global_p90]
        
        anomaly_count = int(inc_anoms.shape[0])
        if anomaly_count > 0:
            earliest_anomaly_ts = inc_anoms['ts'].min().tz_convert(tz)
            earliest_anomaly_ts_str = str(earliest_anomaly_ts)
        else:
            earliest_anomaly_ts_str = ''
        
        if global_p90 == 0:
            severity_ratio = None
        else:
            if np.isnan(incident_max_value):
                severity_ratio = None
            else:
                severity_ratio = float(incident_max_value) / float(global_p90)
        
        records_relaxed.append({
            'component': comp,
            'kpi_name': kpi,
            'overall_count': overall_count,
            'incident_count': incident_count,
            'global_p90': global_p90,
            'anomaly_count': anomaly_count,
            'earliest_anomaly_ts': earliest_anomaly_ts_str,
            'incident_max_value': incident_max_value,
            'severity_ratio': severity_ratio
        })
        if anomaly_count > 0:
            ar = inc_anoms.loc[:, ['timestamp','cmdb_id','kpi_name','value','ts']].copy()
            ar['global_p90'] = global_p90
            anomaly_rows_relaxed.append(ar)
    if anomaly_rows_relaxed:
        metric_anomalies_relaxed = pd.concat(anomaly_rows_relaxed, ignore_index=True)
    else:
        metric_anomalies_relaxed = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value','ts','global_p90'])
    metric_anomalies_relaxed_summary = pd.DataFrame.from_records(records_relaxed)
    metric_anomalies_relaxed_summary['severity_sort'] = metric_anomalies_relaxed_summary['severity_ratio'].apply(lambda x: -np.inf if x is None else x)
    metric_anomalies_relaxed_sorted = metric_anomalies_relaxed_summary.sort_values(
        by=['severity_sort','anomaly_count'],
        ascending=[False, False]
    ).drop(columns=['severity_sort']).reset_index(drop=True)
    metric_anomalies_relaxed_top20 = metric_anomalies_relaxed_sorted.head(20).loc[:, [
        'component','kpi_name','overall_count','incident_count','global_p90',
        'anomaly_count','earliest_anomaly_ts','incident_max_value','severity_ratio'
    ]]

# Keep metric_anomalies (row-level) and metric_anomalies_top20, metric_anomalies_relaxed_top20 in kernel
metric_anomalies_top20, metric_anomalies.shape, metric_anomalies_top20.shape, metric_anomalies_relaxed_top20.head(20)

```
Out[3]:
```
Summary of metric anomaly detection (incident window 2021-03-10 07:00:00 — 2021-03-10 07:30:00 UTC+8):

- Overall
  - Row-level anomalies detected and stored in kernel variable metric_anomalies (8513 rows).
  - Per-component+KPI summary stored in metric_anomalies_top20 (top 20 by severity_ratio shown below).
  - Since anomalies were found using global P95, a relaxed P90 table was not required.

- Top anomaly records (sorted by severity_ratio; top 5 of the reported top-20):
  1. Mysql01 — Mysql-MySQL_3306_Innodb Row Lock Time
     - overall_count: 21, incident_count: 21
     - global P95: 0.1
     - anomaly_count: 3, earliest anomaly: 2021-03-10 07:06:00+08:00
     - incident_max_value: 3.459, severity_ratio: 34.59
  2. Mysql02 — Mysql-MySQL_3306_Opened Tables
     - overall_count: 21, incident_count: 21
     - global P95: 0.1356
     - anomaly_count: 2, earliest anomaly: 2021-03-10 07:26:00+08:00
     - incident_max_value: 0.8033, severity_ratio: 5.92
  3. Mysql02 — Mysql-MySQL_3306_Table open cache misses
     - overall_count: 21, incident_count: 21
     - global P95: 0.1356
     - anomaly_count: 2, earliest anomaly: 2021-03-10 07:26:00+08:00
     - incident_max_value: 0.8033, severity_ratio: 5.92
  4. apache02 — OSLinux-LOCALDISK-sda_DSKBps
     - overall_count: 16, incident_count: 16
     - global P95: 1.25
     - anomaly_count: 1, earliest anomaly: 2021-03-10 07:18:00+08:00
     - incident_max_value: 5.0, severity_ratio: 4.00
  5. apache02 — OSLinux-LOCALDISK-sdb_DSKPercent...
     - overall_count: 16, incident_count: 16
     - global P95: 0.2
     - anomaly_count: 1, earliest anomaly: 2021-03-10 07:12:00+08:00
     - incident_max_value: 0.8, severity_ratio: 4.00

- Observations & suggested next steps
  - The highest severity anomalies are MySQL-related (notably InnoDB Row Lock Time on Mysql01 with severity_ratio >> 1). This indicates row-lock-related contention spikes during the incident window and is likely important for root-cause investigation.
  - Several disk I/O metrics (DSKRead/DSKBps/DSKRTps) on IG01, Mysql01, Mysql02 and apache02 also show elevated values relative to their global P95 (severity_ratio ≈ 2–3.3), suggesting elevated disk I/O during the window.
  - Recommend drilling down into the full anomaly rows (metric_anomalies) for timestamps and surrounding context, and correlating with traces/logs for the same timestamps to pinpoint the cause (e.g., heavy queries, backups, or disk saturation).

If you want, I can:
- show the full metric_anomalies schema and a few representative anomaly rows, or
- produce the complete top-20 table as a compact dataframe for copy/paste.

The original code execution output of IPython Kernel is also provided below for reference:

(   component                                           kpi_name  overall_count  incident_count  global_p95  anomaly_count        earliest_anomaly_ts  incident_max_value  severity_ratio
0    Mysql01              Mysql-MySQL_3306_Innodb Row Lock Time             21              21    0.100000              3  2021-03-10 07:06:00+08:00            3.459000       34.590000
1    Mysql02                     Mysql-MySQL_3306_Opened Tables             21              21    0.135600              2  2021-03-10 07:26:00+08:00            0.803300        5.924041
2    Mysql02           Mysql-MySQL_3306_Table open cache misses             21              21    0.135600              2  2021-03-10 07:26:00+08:00            0.803300        5.924041
3   apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps             16              16    1.250000              1  2021-03-10 07:18:00+08:00            5.000000        4.000000
4   apache02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...             16              16    0.200000              1  2021-03-10 07:12:00+08:00            0.800000        4.000000
5    Mysql02              Mysql-MySQL_3306_Innodb Row Lock Time             21              21    0.950000              2  2021-03-10 07:12:00+08:00            3.600000        3.789474
6       IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps             15              15    0.245010              1  2021-03-10 07:19:00+08:00            0.816700        3.333333
7    Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps             15              15    0.010000              1  2021-03-10 07:08:00+08:00            0.033333        3.333333
8       IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps             15              15    0.600000              1  2021-03-10 07:13:00+08:00            2.000000        3.333333
9       IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead             15              15   14.319990              1  2021-03-10 07:19:00+08:00           47.733300        3.333333
10      IG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps             15              15    0.060000              1  2021-03-10 07:00:00+08:00            0.200000        3.333333
11      IG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead             15              15    1.800000              1  2021-03-10 07:00:00+08:00            6.000000        3.333333
12   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead             15              15    0.120000              1  2021-03-10 07:08:00+08:00            0.400000        3.333333
13   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps             15              15    0.900000              1  2021-03-10 07:25:00+08:00            3.000000        3.333333
14   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...             15              15    0.240000              1  2021-03-10 07:19:00+08:00            0.800000        3.333333
15   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...             15              15    2.400000              1  2021-03-10 07:05:00+08:00            8.000000        3.333333
16   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps             15              15    0.060000              1  2021-03-10 07:25:00+08:00            0.200000        3.333333
17   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead             15              15    6.720000              1  2021-03-10 07:25:00+08:00           22.400000        3.333333
18   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps             15              15    0.376667              1  2021-03-10 07:17:00+08:00            1.100000        2.920352
19   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead             15              15    5.773343              1  2021-03-10 07:17:00+08:00           14.266700        2.471133, (8513, 6), (20, 9), Empty DataFrame
Columns: []
Index: [])```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing variables: metric_anomalies, full_metric_kpi_stats, tz
# Ensure metric_anomalies has 'ts' as timezone-aware datetime
metric_anomalies['ts'] = pd.to_datetime(metric_anomalies['ts']).dt.tz_convert(tz)

# Sort and group consecutive-minute anomaly rows per component+kpi
metric_anomalies_sorted = metric_anomalies.sort_values(['cmdb_id','kpi_name','ts']).reset_index(drop=True)

def assign_sequence_ids(df):
    # compute diff in seconds between consecutive ts
    df = df.copy()
    df['diff_s'] = df['ts'].diff().dt.total_seconds().fillna(0)
    # start new sequence when diff != 60
    df['seq_id'] = (df['diff_s'] != 60).cumsum()
    return df

seq_dfs = []
for (comp, kpi), grp in metric_anomalies_sorted.groupby(['cmdb_id','kpi_name'], sort=False):
    seq_dfs.append(assign_sequence_ids(grp))

seq_all = pd.concat(seq_dfs, ignore_index=True)

# Aggregate sequences into faults
faults = seq_all.groupby(['cmdb_id','kpi_name','seq_id']).agg(
    fault_start_ts=('ts','min'),
    fault_end_ts=('ts','max'),
    point_count=('ts','count'),
    fault_max_value=('value','max')
).reset_index(drop=False)

# Map overall_count and global_p95 from full_metric_kpi_stats
map_stats = full_metric_kpi_stats[['component','kpi_name','count','p95']].rename(
    columns={'component':'cmdb_id','count':'overall_count','p95':'global_p95'}
)
faults = faults.merge(map_stats, on=['cmdb_id','kpi_name'], how='left')

# Compute severity_ratio (None when global_p95 == 0)
def compute_severity(row):
    gp95 = row['global_p95']
    fm = row['fault_max_value']
    if pd.isna(gp95) or gp95 == 0:
        return None
    if pd.isna(fm):
        return None
    return float(fm) / float(gp95)

faults['severity_ratio'] = faults.apply(compute_severity, axis=1)
faults['earliest_anomaly_ts'] = faults['fault_start_ts'].dt.tz_convert(tz).astype(str)

# Filter out noise: remove groups where point_count == 1 AND fault_max_value <= global_p95 * 1.5
def is_noise(row):
    if row['point_count'] != 1:
        return False
    gp95 = row['global_p95']
    fm = row['fault_max_value']
    # if gp95 is nan, treat as not noise (keep)
    if pd.isna(gp95):
        return False
    # gp95 == 0: threshold 0, so drop single point if fm <= 0 (i.e., non-positive)
    threshold = gp95 * 1.5
    return (fm <= threshold)

faults['is_noise'] = faults.apply(is_noise, axis=1)
metric_faults = faults[~faults['is_noise']].copy()

# Select and format required columns
metric_faults = metric_faults.rename(columns={'cmdb_id':'component'}).loc[:, [
    'component','kpi_name','fault_start_ts','fault_end_ts','point_count',
    'fault_max_value','overall_count','global_p95','severity_ratio','earliest_anomaly_ts'
]]

# Sort by severity_ratio desc (None last), then by point_count desc
metric_faults['severity_sort'] = metric_faults['severity_ratio'].apply(lambda x: -np.inf if x is None else x)
metric_faults_sorted = metric_faults.sort_values(by=['severity_sort','point_count'], ascending=[False, False]).drop(columns=['severity_sort']).reset_index(drop=True)

# Top 20 faults for compact display
metric_faults_top20 = metric_faults_sorted.head(20)

# Keep full faults table in kernel variable metric_faults (already stored) and show top20 and shape
metric_faults_top20, metric_faults.shape

```
Out[4]:
```
Summary of detected metric faults (incident window 2021-03-10 07:00–07:30 UTC+8):

- Overview
  - Full faults table saved in kernel variable metric_faults (1071 fault groups).
  - The top 20 faults (sorted by severity_ratio) show several clear hotspots: MySQL row-locking and multiple disk I/O spikes across Mysql02, IG01, apache02, and Mysql01.
  - Single-minute spikes that were considered small noise (point_count == 1 and value <= 1.5 * global_p95) were filtered out.

- Top fault highlights (top 5)
  1. Mysql01 — "Mysql-MySQL_3306_Innodb Row Lock Time"
     - Fault span: 2021-03-10 07:06:00
     - point_count: 1
     - fault_max_value: 3.459
     - global P95: 0.1
     - severity_ratio ≈ 34.6 (very high)
  2. Mysql02 — "Mysql-MySQL_3306_Opened Tables"
     - Fault span: 2021-03-10 07:26:00 — 07:27:00
     - point_count: 2
     - fault_max_value: 0.8033
     - global P95: 0.1356
     - severity_ratio ≈ 5.92
  3. Mysql02 — "Mysql-MySQL_3306_Table open cache misses"
     - Fault span: 2021-03-10 07:26:00 — 07:27:00
     - point_count: 2
     - fault_max_value: 0.8033
     - global P95: 0.1356
     - severity_ratio ≈ 5.92
  4. apache02 — disk throughput (sda DSKBps)
     - Fault at 2021-03-10 07:18:00 (point_count 1)
     - fault_max_value: 5.0 vs P95 1.25 → severity_ratio = 4.0
  5. apache02 — other local-disk metric (sdb DSKPercent...)
     - Fault at 2021-03-10 07:12:00 (point_count 1)
     - fault_max_value: 0.8 vs P95 0.2 → severity_ratio = 4.0

- Other pattern notes
  - Multiple disk I/O metrics (DSKRead, DSKRTps, DSKBps) on IG01, Mysql01, Mysql02 and apache02 show elevated faults with severity_ratio ≈ 2–3.3.
  - Many faults are single-minute bursts; others (notably the Mysql02 table/opened-tables events) span 2 consecutive minutes.
  - The most severe single alarm is the InnoDB row-lock spike on Mysql01 (severity_ratio >> 1), which strongly suggests contention/locking issues at that timestamp.

- Recommended next steps
  1. Inspect MySQL at the given timestamps (07:06 and ~07:12–07:27) for long-running queries, locks, or maintenance jobs; check slow query logs and InnoDB lock diagnostics.
  2. Investigate disk I/O activity on Mysql02, IG01 and apache02 around the fault timestamps — check I/O wait, concurrent jobs (backups, large queries), and host-level processes.
  3. Correlate these fault timestamps with trace and log anomalies (trace and log datasets) to find affected services/requests.
  4. If desired, I can show a few representative row-level anomalies from metric_anomalies or extract the top-20 faults as a compact dataframe for copying.

If you want, I can export the top-20 fault table (the exact columns you requested) here for quick review.

The original code execution output of IPython Kernel is also provided below for reference:

(   component                                           kpi_name            fault_start_ts              fault_end_ts  point_count  fault_max_value  overall_count  global_p95  severity_ratio        earliest_anomaly_ts
0    Mysql01              Mysql-MySQL_3306_Innodb Row Lock Time 2021-03-10 07:06:00+08:00 2021-03-10 07:06:00+08:00            1         3.459000             21    0.100000       34.590000  2021-03-10 07:06:00+08:00
1    Mysql02                     Mysql-MySQL_3306_Opened Tables 2021-03-10 07:26:00+08:00 2021-03-10 07:27:00+08:00            2         0.803300             21    0.135600        5.924041  2021-03-10 07:26:00+08:00
2    Mysql02           Mysql-MySQL_3306_Table open cache misses 2021-03-10 07:26:00+08:00 2021-03-10 07:27:00+08:00            2         0.803300             21    0.135600        5.924041  2021-03-10 07:26:00+08:00
3   apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps 2021-03-10 07:18:00+08:00 2021-03-10 07:18:00+08:00            1         5.000000             16    1.250000        4.000000  2021-03-10 07:18:00+08:00
4   apache02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer... 2021-03-10 07:12:00+08:00 2021-03-10 07:12:00+08:00            1         0.800000             16    0.200000        4.000000  2021-03-10 07:12:00+08:00
5    Mysql02              Mysql-MySQL_3306_Innodb Row Lock Time 2021-03-10 07:12:00+08:00 2021-03-10 07:12:00+08:00            1         3.600000             21    0.950000        3.789474  2021-03-10 07:12:00+08:00
6       IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-10 07:19:00+08:00 2021-03-10 07:19:00+08:00            1         0.816700             15    0.245010        3.333333  2021-03-10 07:19:00+08:00
7    Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-10 07:08:00+08:00 2021-03-10 07:08:00+08:00            1         0.033333             15    0.010000        3.333333  2021-03-10 07:08:00+08:00
8       IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps 2021-03-10 07:13:00+08:00 2021-03-10 07:13:00+08:00            1         2.000000             15    0.600000        3.333333  2021-03-10 07:13:00+08:00
9       IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead 2021-03-10 07:19:00+08:00 2021-03-10 07:19:00+08:00            1        47.733300             15   14.319990        3.333333  2021-03-10 07:19:00+08:00
10      IG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-10 07:00:00+08:00 2021-03-10 07:00:00+08:00            1         0.200000             15    0.060000        3.333333  2021-03-10 07:00:00+08:00
11      IG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead 2021-03-10 07:00:00+08:00 2021-03-10 07:00:00+08:00            1         6.000000             15    1.800000        3.333333  2021-03-10 07:00:00+08:00
12   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead 2021-03-10 07:08:00+08:00 2021-03-10 07:08:00+08:00            1         0.400000             15    0.120000        3.333333  2021-03-10 07:08:00+08:00
13   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps 2021-03-10 07:25:00+08:00 2021-03-10 07:25:00+08:00            1         3.000000             15    0.900000        3.333333  2021-03-10 07:25:00+08:00
14   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer... 2021-03-10 07:19:00+08:00 2021-03-10 07:19:00+08:00            1         0.800000             15    0.240000        3.333333  2021-03-10 07:19:00+08:00
15   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer... 2021-03-10 07:05:00+08:00 2021-03-10 07:05:00+08:00            1         8.000000             15    2.400000        3.333333  2021-03-10 07:05:00+08:00
16   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps 2021-03-10 07:25:00+08:00 2021-03-10 07:25:00+08:00            1         0.200000             15    0.060000        3.333333  2021-03-10 07:25:00+08:00
17   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead 2021-03-10 07:25:00+08:00 2021-03-10 07:25:00+08:00            1        22.400000             15    6.720000        3.333333  2021-03-10 07:25:00+08:00
18   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-10 07:17:00+08:00 2021-03-10 07:17:00+08:00            1         1.100000             15    0.376667        2.920352  2021-03-10 07:17:00+08:00
19   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead 2021-03-10 07:17:00+08:00 2021-03-10 07:17:00+08:00            1        14.266700             15    5.773343        2.471133  2021-03-10 07:17:00+08:00, (1071, 11))```
```

In [ ]:
# In[5]:

# Reuse existing variables: trace_df, full_trace_kpi_stats, tz, incident_start, incident_end
import numpy as np
import pandas as pd

# Ensure trace_df 'ts' is timezone-aware
trace_df['ts'] = pd.to_datetime(trace_df['ts']).dt.tz_convert(tz)

# Select component+trace entries that have incident_count > 0
trace_kpis_incident = full_trace_kpi_stats[full_trace_kpi_stats['incident_count'] > 0].copy()

records = []
anomaly_rows = []

for _, r in trace_kpis_incident.iterrows():
    comp = r['component']
    trace_name = r['kpi_name']
    overall_count = int(r['count'])
    incident_count = int(r['incident_count'])
    global_p95 = float(r['p95'])
    
    # incident-window rows for this component+trace
    inc_rows = trace_df[
        (trace_df['cmdb_id'] == comp) &
        (trace_df['trace_name'] == trace_name) &
        (trace_df['ts'] >= incident_start) &
        (trace_df['ts'] <= incident_end)
    ].copy()
    
    if inc_rows.empty:
        incident_max_value = np.nan
    else:
        incident_max_value = float(inc_rows['value'].max())
    
    # anomaly rule: if global_p95 == 0 -> anomaly if value > 0; else value >= global_p95
    if global_p95 == 0.0:
        inc_anoms = inc_rows[inc_rows['value'] > 0]
    else:
        inc_anoms = inc_rows[inc_rows['value'] >= global_p95]
    
    anomaly_count = int(inc_anoms.shape[0])
    if anomaly_count > 0:
        earliest_anom_ts = str(inc_anoms['ts'].min().tz_convert(tz))
    else:
        earliest_anom_ts = ''
    
    if global_p95 == 0:
        severity_ratio = None
    else:
        if np.isnan(incident_max_value):
            severity_ratio = None
        else:
            severity_ratio = float(incident_max_value) / float(global_p95)
    
    records.append({
        'component': comp,
        'trace_name': trace_name,
        'overall_count': overall_count,
        'incident_count': incident_count,
        'global_p95': global_p95,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_ts': earliest_anom_ts,
        'incident_max_value': incident_max_value,
        'severity_ratio': severity_ratio
    })
    
    if anomaly_count > 0:
        ar = inc_anoms.loc[:, ['timestamp','cmdb_id','trace_name','value','ts']].copy()
        ar['global_p95'] = global_p95
        anomaly_rows.append(ar)

# Concatenate row-level anomalies
if anomaly_rows:
    trace_anomalies = pd.concat(anomaly_rows, ignore_index=True)
else:
    trace_anomalies = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','ts','global_p95'])

# Summary table per component+trace
trace_anomalies_summary = pd.DataFrame.from_records(records)

# Sort by severity_ratio desc (None last) then anomaly_count desc
trace_anomalies_summary['severity_sort'] = trace_anomalies_summary['severity_ratio'].apply(lambda x: -np.inf if x is None else x)
trace_anomalies_sorted = trace_anomalies_summary.sort_values(by=['severity_sort','anomaly_count'], ascending=[False, False]).drop(columns=['severity_sort']).reset_index(drop=True)

# Top 20 results for compact display
trace_anomalies_top20 = trace_anomalies_sorted.head(20).loc[:, [
    'component','trace_name','overall_count','incident_count','global_p95',
    'anomaly_count','earliest_anomaly_ts','incident_max_value','severity_ratio'
]]

# Return compact outputs: top20 and shape of full anomalies table (stored in trace_anomalies)
trace_anomalies_top20, trace_anomalies.shape

```
Out[5]:
```
Summary of trace anomaly detection (incident window 2021-03-10 07:00–07:30 UTC+8):

- Overall
  - Row-level trace anomalies saved in kernel variable trace_anomalies (502 rows).
  - Per-component+trace summary produced; top-20 records (by severity_ratio) are shown below.

- Top anomaly highlights (top entries)
  1. MG01 — trace.from_dockerB1.duration_p95
     - overall_count: 30, incident_count: 30
     - global P95: 0.019605, anomaly_count: 2
     - earliest anomaly: 2021-03-10 07:12:00+08:00
     - incident_max_value: 0.2658, severity_ratio ≈ 13.56
  2. dockerB1 — trace.to_MG01.duration_p95
     - same stats as above (paired inter-service trace), severity_ratio ≈ 13.56
  3. MG02 — trace.from_dockerA1.duration_p95
     - overall_count: 29, incident_count: 29, global P95: 0.01368
     - anomaly_count: 2, earliest anomaly: 2021-03-10 07:21:00+08:00
     - incident_max_value: 0.0914, severity_ratio ≈ 6.68
  4. dockerA1 — trace.to_MG02.duration_p95
     - paired entry for MG02 above, severity_ratio ≈ 6.68
  5. MG02 — trace.from_dockerB1.duration_p95 / dockerB1 → MG02
     - global P95 ≈ 0.01485, incident_max ≈ 0.08625, severity_ratio ≈ 5.81
  - Several other MG02/MG01 ↔ dockerA*/dockerB* traces show elevated p95-duration spikes (severity ratios between ~1.9 and ~5.8).
  - A few duration_mean metrics (same routes) also exceed their p95 with lower severity_ratios (~2.1–2.75).
  - Tomcat01 ↔ MG01 and IG02 ↔ Tomcat03 p95 traces show moderate increases (severity_ratio ≈ 1.8–1.9).

- Pattern & interpretation
  - Most anomalous trace series are inter-service latency p95 spikes (MG01/MG02 with various docker nodes and Tomcat), typically with incident_count = full incident coverage (29–30 samples) but only a small number of anomaly samples (anomaly_count = 2).
  - Highest severity traces are MG01 related (large multiples of global P95), indicating brief but substantial latency spikes on those routes around the listed earliest_anomaly_ts (notably ~07:12 and ~07:21).
  - These trace anomalies align with service-to-service latency issues rather than broad constant degradations.

- Recommended next steps
  1. Correlate top trace anomaly timestamps (e.g., 07:12, 07:21) with metric faults (metric_faults) and log events to find causation (CPU, disk, DB locks).
  2. Inspect the affected services (MG01/MG02, dockerA/B nodes, Tomcat01/03) at those timestamps for resource pressure, request spikes, or network issues.
  3. Use trace_anomalies (stored) to extract the detailed row-level anomalies for deeper correlation and root-cause analysis.

If you want, I can produce a compact side-by-side correlation of the top trace faults with metric_faults for the same timestamps.

The original code execution output of IPython Kernel is also provided below for reference:

(   component                         trace_name  overall_count  incident_count  global_p95  anomaly_count        earliest_anomaly_ts  incident_max_value  severity_ratio
0       MG01   trace.from_dockerB1.duration_p95             30              30    0.019605              2  2021-03-10 07:12:00+08:00            0.265800       13.557766
1   dockerB1         trace.to_MG01.duration_p95             30              30    0.019605              2  2021-03-10 07:12:00+08:00            0.265800       13.557766
2       MG02   trace.from_dockerA1.duration_p95             29              29    0.013680              2  2021-03-10 07:21:00+08:00            0.091400        6.681287
3   dockerA1         trace.to_MG02.duration_p95             29              29    0.013680              2  2021-03-10 07:21:00+08:00            0.091400        6.681287
4       MG02   trace.from_dockerB1.duration_p95             30              30    0.014852              2  2021-03-10 07:14:00+08:00            0.086250        5.807103
5   dockerB1         trace.to_MG02.duration_p95             30              30    0.014852              2  2021-03-10 07:14:00+08:00            0.086250        5.807103
6       MG02   trace.from_dockerA2.duration_p95             27              27    0.020970              2  2021-03-10 07:04:00+08:00            0.118550        5.653314
7   dockerA2         trace.to_MG02.duration_p95             27              27    0.020970              2  2021-03-10 07:04:00+08:00            0.118550        5.653314
8       MG02  trace.from_dockerA1.duration_mean             29              29    0.008040              2  2021-03-10 07:21:00+08:00            0.022111        2.750124
9   dockerA1        trace.to_MG02.duration_mean             29              29    0.008040              2  2021-03-10 07:21:00+08:00            0.022111        2.750124
10      MG02  trace.from_dockerA2.duration_mean             27              27    0.010540              2  2021-03-10 07:04:00+08:00            0.027800        2.637571
11  dockerA2        trace.to_MG02.duration_mean             27              27    0.010540              2  2021-03-10 07:04:00+08:00            0.027800        2.637571
12      MG02  trace.from_dockerB1.duration_mean             30              30    0.012239              2  2021-03-10 07:09:00+08:00            0.026500        2.165289
13  dockerB1        trace.to_MG02.duration_mean             30              30    0.012239              2  2021-03-10 07:09:00+08:00            0.026500        2.165289
14      MG01   trace.from_dockerB2.duration_p95             30              30    0.030185              2  2021-03-10 07:06:00+08:00            0.058100        1.924797
15  dockerB2         trace.to_MG01.duration_p95             30              30    0.030185              2  2021-03-10 07:06:00+08:00            0.058100        1.924797
16      MG01   trace.from_Tomcat01.duration_p95             30              30    0.588375              2  2021-03-10 07:05:00+08:00            1.101750        1.872530
17  Tomcat01         trace.to_MG01.duration_p95             30              30    0.588375              2  2021-03-10 07:05:00+08:00            1.101750        1.872530
18      IG02     trace.to_Tomcat03.duration_p95             30              30    0.806490              2  2021-03-10 07:23:00+08:00            1.497800        1.857184
19  Tomcat03       trace.from_IG02.duration_p95             30              30    0.806490              2  2021-03-10 07:23:00+08:00            1.497800        1.857184, (502, 6))```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse existing kernel variables: metric_faults, trace_anomalies, full_trace_kpi_stats, tz
# Ensure datetime types are timezone-aware (Asia/Shanghai)
metric_faults['fault_start_ts'] = pd.to_datetime(metric_faults['fault_start_ts']).dt.tz_convert(tz)
metric_faults['fault_end_ts'] = pd.to_datetime(metric_faults['fault_end_ts']).dt.tz_convert(tz)

# Build trace-level summary from row-level trace_anomalies:
# earliest anomaly ts per (cmdb_id, trace_name), incident_max_value, and map global_p95
if not trace_anomalies.empty:
    trace_anomalies['ts'] = pd.to_datetime(trace_anomalies['ts']).dt.tz_convert(tz)
    trace_summary = trace_anomalies.groupby(['cmdb_id','trace_name']).agg(
        trace_earliest_anomaly_ts=('ts','min'),
        trace_incident_max_value=('value','max')
    ).reset_index()
else:
    trace_summary = pd.DataFrame(columns=['cmdb_id','trace_name','trace_earliest_anomaly_ts','trace_incident_max_value'])

# Map global_p95 and compute trace severity_ratio using full_trace_kpi_stats
trace_p95_map = full_trace_kpi_stats.rename(columns={'component':'cmdb_id','kpi_name':'trace_name'})[['cmdb_id','trace_name','p95']]
trace_summary = trace_summary.merge(trace_p95_map, on=['cmdb_id','trace_name'], how='left')
trace_summary = trace_summary.rename(columns={'p95':'trace_global_p95'})

def compute_trace_sev(row):
    gp95 = row['trace_global_p95']
    mv = row['trace_incident_max_value']
    if pd.isna(gp95) or gp95 == 0:
        return None
    if pd.isna(mv):
        return None
    return float(mv) / float(gp95)

trace_summary['trace_severity_ratio'] = trace_summary.apply(compute_trace_sev, axis=1)

# Prepare metric faults: select top 50 by metric severity_ratio (desc), treat None as very small
mf = metric_faults.copy()
mf['severity_sort'] = mf['severity_ratio'].apply(lambda x: -np.inf if x is None else x)
mf_top50 = mf.sort_values(by=['severity_sort','fault_start_ts'], ascending=[False, True]).head(50).drop(columns=['severity_sort']).reset_index(drop=True)

# Correlate: for each metric fault find trace_summary rows whose earliest_anomaly_ts within [fault_start_ts - 60s, fault_end_ts + 60s]
corr_rows = []
for _, f in mf_top50.iterrows():
    f_start = pd.to_datetime(f['fault_start_ts']).tz_convert(tz)
    f_end = pd.to_datetime(f['fault_end_ts']).tz_convert(tz)
    window_start = f_start - pd.Timedelta(seconds=60)
    window_end = f_end + pd.Timedelta(seconds=60)
    
    # find matches
    matches = trace_summary[
        (trace_summary['trace_earliest_anomaly_ts'] >= window_start) &
        (trace_summary['trace_earliest_anomaly_ts'] <= window_end)
    ].copy()
    
    # If multiple matches, keep up to 3 highest trace_severity_ratio (None -> -inf)
    if not matches.empty:
        matches['trace_sev_sort'] = matches['trace_severity_ratio'].apply(lambda x: -np.inf if x is None else x)
        matches = matches.sort_values(by=['trace_sev_sort'], ascending=False).head(3).drop(columns=['trace_sev_sort'])
        # create a correlation row for each match
        for _, m in matches.iterrows():
            corr_rows.append({
                'metric_component': f['component'],
                'metric_kpi_name': f['kpi_name'],
                'fault_start_ts': f_start,
                'fault_end_ts': f_end,
                'point_count': int(f['point_count']),
                'fault_max_value': float(f['fault_max_value']),
                'metric_global_p95': float(f['global_p95']) if not pd.isna(f['global_p95']) else None,
                'metric_severity_ratio': f['severity_ratio'],
                'matched_trace_component': m['cmdb_id'],
                'trace_name': m['trace_name'],
                'trace_earliest_anomaly_ts': m['trace_earliest_anomaly_ts'],
                'trace_incident_max_value': m['trace_incident_max_value'],
                'trace_global_p95': m['trace_global_p95'],
                'trace_severity_ratio': m['trace_severity_ratio']
            })
    else:
        # no match: include a single row with trace fields empty
        corr_rows.append({
            'metric_component': f['component'],
            'metric_kpi_name': f['kpi_name'],
            'fault_start_ts': f_start,
            'fault_end_ts': f_end,
            'point_count': int(f['point_count']),
            'fault_max_value': float(f['fault_max_value']),
            'metric_global_p95': float(f['global_p95']) if not pd.isna(f['global_p95']) else None,
            'metric_severity_ratio': f['severity_ratio'],
            'matched_trace_component': None,
            'trace_name': None,
            'trace_earliest_anomaly_ts': pd.NaT,
            'trace_incident_max_value': None,
            'trace_global_p95': None,
            'trace_severity_ratio': None
        })

# Full correlation table stored in kernel variable
metric_trace_correlation = pd.DataFrame.from_records(corr_rows)

# Convert timestamps to string for compact display, keep full datetimes in stored variable
metric_trace_correlation_display = metric_trace_correlation.copy()
metric_trace_correlation_display['fault_start_ts'] = metric_trace_correlation_display['fault_start_ts'].astype(str)
metric_trace_correlation_display['fault_end_ts'] = metric_trace_correlation_display['fault_end_ts'].astype(str)
metric_trace_correlation_display['trace_earliest_anomaly_ts'] = metric_trace_correlation_display['trace_earliest_anomaly_ts'].astype(str).replace('NaT','')

# Sort display by metric_severity_ratio desc then fault_start_ts asc
metric_trace_correlation_display['metric_sev_sort'] = metric_trace_correlation_display['metric_severity_ratio'].apply(lambda x: -np.inf if x is None else x)
metric_trace_correlation_display = metric_trace_correlation_display.sort_values(by=['metric_sev_sort','fault_start_ts'], ascending=[False, True]).drop(columns=['metric_sev_sort']).reset_index(drop=True)

# Limit to 20 rows for output
metric_trace_correlation_top20 = metric_trace_correlation_display.head(20)

# Aggregate summary:
# For each unique metric fault (by metric_component + metric_kpi_name + fault_start_ts), determine if it has matched traces and whether any matched trace is before fault_start_ts
agg = metric_trace_correlation.copy()
# mark has_match
agg['has_match'] = ~agg['matched_trace_component'].isna()
# compare trace time with fault_start_ts
def cmp_before(row):
    t = row['trace_earliest_anomaly_ts']
    if pd.isna(t) or (t is pd.NaT):
        return 'no_match'
    try:
        tdt = pd.to_datetime(t)
    except Exception:
        return 'no_match'
    if tdt < row['fault_start_ts']:
        return 'before'
    else:
        return 'after_or_equal'

agg['relation'] = agg.apply(cmp_before, axis=1)

# group per metric fault identity
fault_groups = agg.groupby(['metric_component','metric_kpi_name','fault_start_ts']).agg(
    any_match=('has_match','any'),
    any_before=('relation', lambda x: (x=='before').any()),
    any_after_or_equal=('relation', lambda x: (x=='after_or_equal').any())
).reset_index()

count_before = int(fault_groups['any_before'].sum())
count_after_or_equal = int(fault_groups['any_after_or_equal'].sum())
count_no_match = int((~fault_groups['any_match']).sum())

aggregate_summary = {
    'metric_faults_with_match_before_fault_start': count_before,
    'metric_faults_with_match_after_or_equal_fault_start': count_after_or_equal,
    'metric_faults_with_no_matched_traces': count_no_match
}

# Final compact outputs: top 20 correlation rows and the aggregate summary; full correlation table stored as metric_trace_correlation
metric_trace_correlation_top20, aggregate_summary, metric_trace_correlation.shape

```
Out[6]:
```
Summary of metric ↔ trace correlation (incident window ±1 minute):

- Correlation coverage
  - Full correlation table stored in kernel variable: metric_trace_correlation (145 correlation rows).
  - Top 20 correlated rows shown (one row per metric–trace match, up to 3 matches per metric fault).

- Aggregate counts (per unique metric fault):
  - Metric faults with at least one matched trace anomaly occurring before the fault start: 24
  - Metric faults with matched trace anomaly at or after the fault start: 36
  - Metric faults with no matched traces within ±1 minute: 2

- Key correlated findings (high-level highlights from the top matches)
  - Strong link between MySQL row-lock spikes and MG01/MG02 trace spikes:
    - Mysql01 InnoDB Row Lock Time (07:06) matches MG01 traces (trace.from_dockerB2.duration_p95 and trace.to_MG01.duration_p95) at 07:06, and a Tomcat01→MG01 trace at 07:05 — traces sometimes occur slightly before the DB fault.
    - Mysql02 InnoDB/Opened-Tables/Table-open-cache spikes (~07:12 and ~07:26) match MG01/MG02 related traces at the same timestamps (trace.duration_p95 on MG nodes and docker-to-MG traces).
  - Disk I/O metric spikes on apache02 / IG01 / Mysql nodes (DSKBps, DSKRead, DSKRTps) often correlate with nearby trace anomalies on MG/docker routes (e.g., apache02 sdb DSKPercent at 07:12 aligns with MG01 trace.from_dockerB1.duration_p95 at 07:12).
  - Many correlated trace anomalies are inter-service latency p95 spikes (MG ↔ docker/Tomcat), often occurring just before or concurrent with metric faults on DB/disk metrics — suggests upstream latency possibly triggering DB contention or I/O bursts.

- Interpretation / next steps
  1. The strongest signal: MySQL contention (InnoDB row lock time) aligns with elevated inter-service latency on MG nodes at the same times — investigate requests routed through MG01/MG02 around 07:05–07:27.
  2. Check MySQL slow queries, lock tables, and connection activity at the listed times (07:06, 07:12, 07:26); collect InnoDB lock diagnostics and slow query logs.
  3. For correlated disk-I/O spikes, inspect host-level I/O activity and processes (backups, batch jobs) that might coincide with the trace latency peaks.
  4. Correlate logs (log.csv / error logs) for the matched timestamps to identify offending requests or errors.
  5. If desired, I can:
     - Export the full metric_trace_correlation table, or
     - Produce a focused timeline view for a specific fault (e.g., Mysql01 07:06) showing trace and metric rows around that time.

If you want, tell me which metric fault to inspect first and I’ll extract the matching trace & metric rows for that time window.

The original code execution output of IPython Kernel is also provided below for reference:

(   metric_component                                    metric_kpi_name             fault_start_ts               fault_end_ts  point_count  fault_max_value  metric_global_p95  metric_severity_ratio matched_trace_component                         trace_name  trace_earliest_anomaly_ts  trace_incident_max_value  trace_global_p95  trace_severity_ratio
0           Mysql01              Mysql-MySQL_3306_Innodb Row Lock Time  2021-03-10 07:06:00+08:00  2021-03-10 07:06:00+08:00            1         3.459000             0.1000              34.590000                    MG01   trace.from_dockerB2.duration_p95  2021-03-10 07:06:00+08:00                  0.058100          0.030185              1.924797
1           Mysql01              Mysql-MySQL_3306_Innodb Row Lock Time  2021-03-10 07:06:00+08:00  2021-03-10 07:06:00+08:00            1         3.459000             0.1000              34.590000                dockerB2         trace.to_MG01.duration_p95  2021-03-10 07:06:00+08:00                  0.058100          0.030185              1.924797
2           Mysql01              Mysql-MySQL_3306_Innodb Row Lock Time  2021-03-10 07:06:00+08:00  2021-03-10 07:06:00+08:00            1         3.459000             0.1000              34.590000                Tomcat01         trace.to_MG01.duration_p95  2021-03-10 07:05:00+08:00                  1.101750          0.588375              1.872530
3           Mysql02                     Mysql-MySQL_3306_Opened Tables  2021-03-10 07:26:00+08:00  2021-03-10 07:27:00+08:00            2         0.803300             0.1356               5.924041                dockerA1       trace.from_MG01.duration_p95  2021-03-10 07:26:00+08:00                  0.442150          0.276207              1.600789
4           Mysql02                     Mysql-MySQL_3306_Opened Tables  2021-03-10 07:26:00+08:00  2021-03-10 07:27:00+08:00            2         0.803300             0.1356               5.924041                    MG01     trace.to_dockerA1.duration_p95  2021-03-10 07:26:00+08:00                  0.442150          0.276207              1.600789
5           Mysql02                     Mysql-MySQL_3306_Opened Tables  2021-03-10 07:26:00+08:00  2021-03-10 07:27:00+08:00            2         0.803300             0.1356               5.924041                    MG01      trace.from_dockerB1.row_count  2021-03-10 07:26:00+08:00                 63.000000         47.600000              1.323529
6           Mysql02           Mysql-MySQL_3306_Table open cache misses  2021-03-10 07:26:00+08:00  2021-03-10 07:27:00+08:00            2         0.803300             0.1356               5.924041                dockerA1       trace.from_MG01.duration_p95  2021-03-10 07:26:00+08:00                  0.442150          0.276207              1.600789
7           Mysql02           Mysql-MySQL_3306_Table open cache misses  2021-03-10 07:26:00+08:00  2021-03-10 07:27:00+08:00            2         0.803300             0.1356               5.924041                    MG01     trace.to_dockerA1.duration_p95  2021-03-10 07:26:00+08:00                  0.442150          0.276207              1.600789
8           Mysql02           Mysql-MySQL_3306_Table open cache misses  2021-03-10 07:26:00+08:00  2021-03-10 07:27:00+08:00            2         0.803300             0.1356               5.924041                    MG01      trace.from_dockerB1.row_count  2021-03-10 07:26:00+08:00                 63.000000         47.600000              1.323529
9          apache02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...  2021-03-10 07:12:00+08:00  2021-03-10 07:12:00+08:00            1         0.800000             0.2000               4.000000                    MG01   trace.from_dockerB1.duration_p95  2021-03-10 07:12:00+08:00                  0.265800          0.019605             13.557766
10         apache02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...  2021-03-10 07:12:00+08:00  2021-03-10 07:12:00+08:00            1         0.800000             0.2000               4.000000                dockerB1         trace.to_MG01.duration_p95  2021-03-10 07:12:00+08:00                  0.265800          0.019605             13.557766
11         apache02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...  2021-03-10 07:12:00+08:00  2021-03-10 07:12:00+08:00            1         0.800000             0.2000               4.000000                    MG01      trace.from_dockerA1.row_count  2021-03-10 07:13:00+08:00                 15.000000          9.650000              1.554404
12         apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  2021-03-10 07:18:00+08:00  2021-03-10 07:18:00+08:00            1         5.000000             1.2500               4.000000                    MG01    trace.to_dockerA2.duration_mean  2021-03-10 07:17:00+08:00                  0.199923          0.126568              1.579564
13         apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  2021-03-10 07:18:00+08:00  2021-03-10 07:18:00+08:00            1         5.000000             1.2500               4.000000                dockerA2      trace.from_MG01.duration_mean  2021-03-10 07:17:00+08:00                  0.199923          0.126568              1.579564
14         apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  2021-03-10 07:18:00+08:00  2021-03-10 07:18:00+08:00            1         5.000000             1.2500               4.000000                    IG01    trace.to_Tomcat03.duration_mean  2021-03-10 07:17:00+08:00                  0.180400          0.123078              1.465733
15          Mysql02              Mysql-MySQL_3306_Innodb Row Lock Time  2021-03-10 07:12:00+08:00  2021-03-10 07:12:00+08:00            1         3.600000             0.9500               3.789474                    MG01   trace.from_dockerB1.duration_p95  2021-03-10 07:12:00+08:00                  0.265800          0.019605             13.557766
16          Mysql02              Mysql-MySQL_3306_Innodb Row Lock Time  2021-03-10 07:12:00+08:00  2021-03-10 07:12:00+08:00            1         3.600000             0.9500               3.789474                dockerB1         trace.to_MG01.duration_p95  2021-03-10 07:12:00+08:00                  0.265800          0.019605             13.557766
17          Mysql02              Mysql-MySQL_3306_Innodb Row Lock Time  2021-03-10 07:12:00+08:00  2021-03-10 07:12:00+08:00            1         3.600000             0.9500               3.789474                    MG01      trace.from_dockerA1.row_count  2021-03-10 07:13:00+08:00                 15.000000          9.650000              1.554404
18          Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps  2021-03-10 07:08:00+08:00  2021-03-10 07:08:00+08:00            1         0.033333             0.0100               3.333333                    MG02  trace.from_dockerB1.duration_mean  2021-03-10 07:09:00+08:00                  0.026500          0.012239              2.165289
19          Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps  2021-03-10 07:08:00+08:00  2021-03-10 07:08:00+08:00            1         0.033333             0.0100               3.333333                dockerB1        trace.to_MG02.duration_mean  2021-03-10 07:09:00+08:00                  0.026500          0.012239              2.165289, {'metric_faults_with_match_before_fault_start': 24, 'metric_faults_with_match_after_or_equal_fault_start': 36, 'metric_faults_with_no_matched_traces': 2}, (145, 14))```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# Reuse existing variables from kernel: metric_df, trace_df, tz if available
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Define window (UTC+8)
window_start = pd.Timestamp('2021-03-10 07:04:00').tz_localize(tz)
window_end   = pd.Timestamp('2021-03-10 07:08:00').tz_localize(tz)

# Ensure 'ts' timezone-aware columns exist (they were created earlier). If not, create from 'timestamp'
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    metric_df['ts'] = pd.to_datetime(metric_df['ts']).dt.tz_convert(tz)

if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    trace_df['ts'] = pd.to_datetime(trace_df['ts']).dt.tz_convert(tz)

# Prepare patterns (case-insensitive)
metric_pattern_group1 = r'InnoDB|Innodb|Row Lock'
metric_pattern_group2 = r'LOCALDISK|DSKRead|DSKRTps|DSK\b|DSKBps|DSKRead'  # include common DSK tokens
metric_pattern = rf'({metric_pattern_group1})|({metric_pattern_group2})'

trace_components = ['MG01','dockerB2','Tomcat01','dockerB1']
trace_pattern = r'to_MG01|from_|duration_p95|duration_mean'

# Filter metric rows for Mysql01 and patterns within window
mask_time_metric = (metric_df['ts'] >= window_start) & (metric_df['ts'] <= window_end)
mask_comp_metric = metric_df['cmdb_id'].astype(str) == 'Mysql01'
kpi_series = metric_df['kpi_name'].fillna('').astype(str)

mask_kpi_metric = kpi_series.str.contains(metric_pattern, case=False, regex=True)

metric_selection = metric_df[mask_time_metric & mask_comp_metric & mask_kpi_metric].copy()
metric_selection = metric_selection.sort_values('ts').loc[:, ['timestamp','cmdb_id','kpi_name','value']]

# Also ensure we include any other Mysql01 DSK/LOCALDISK matches (pattern already includes them)
# Limit metric rows to 50
metric_rows_out = metric_selection.head(50).reset_index(drop=True)

# Filter trace rows for selected components and pattern within window
mask_time_trace = (trace_df['ts'] >= window_start) & (trace_df['ts'] <= window_end)
mask_comp_trace = trace_df['cmdb_id'].isin(trace_components)
trace_name_series = trace_df['trace_name'].fillna('').astype(str)
mask_trace_pattern = trace_name_series.str.contains(trace_pattern, case=False, regex=True)

trace_selection = trace_df[mask_time_trace & mask_comp_trace & mask_trace_pattern].copy()
trace_selection = trace_selection.sort_values('ts').loc[:, ['timestamp','cmdb_id','trace_name','value']]

# Limit trace rows to 50
trace_rows_out = trace_selection.head(50).reset_index(drop=True)

# Check for missing requested keys
missing_notes = []
# Check metric component presence
if not (metric_df['cmdb_id'].astype(str) == 'Mysql01').any():
    missing_notes.append("Component 'Mysql01' not found in metric.csv")
else:
    if metric_rows_out.shape[0] == 0:
        missing_notes.append("No Mysql01 metric rows matched the requested InnoDB/Row Lock/LOCALDISK/DSK patterns in the time window")

# Check trace components presence
for comp in trace_components:
    if not (trace_df['cmdb_id'].astype(str) == comp).any():
        missing_notes.append(f"Component '{comp}' not found in trace.csv")

if trace_rows_out.shape[0] == 0:
    missing_notes.append("No trace rows matched the requested components/patterns in the time window")

# Ensure total rows <= 100 (metric limited to 50 and trace to 50 already)
total_rows = len(metric_rows_out) + len(trace_rows_out)
if total_rows > 100:
    # trim trace rows to keep total 100
    allowed_trace = max(0, 100 - len(metric_rows_out))
    trace_rows_out = trace_rows_out.head(allowed_trace).reset_index(drop=True)

# Compact result outputs (metrics first, traces second), keep as DataFrames
metric_rows_out, trace_rows_out, missing_notes

```
Out[7]:
```
Summary of extracted raw rows for 2021-03-10 07:04:00 — 2021-03-10 07:08:00 (UTC+8):

What I returned
- Metrics (50 rows, limited to 50): all rows are for component Mysql01 and match InnoDB-related KPIs (case-insensitive matches for "InnoDB"/"Innodb"/"Row Lock") and LOCALDISK/DSK-related KPIs. Columns returned: timestamp, cmdb_id, kpi_name, value. Examples include InnoDB metrics such as "Innodb os log written" (large value ~742.4), "data written" (~79539.2), "data fsyncs" (~2.4833), "dblwr writes/pages written", "log write requests", etc.
- Traces (50 rows, limited to 50): rows for components MG01, dockerB1, dockerB2 and Tomcat01 matching trace patterns (to_MG01, from_, duration_p95, duration_mean). Columns returned: timestamp, cmdb_id, trace_name, value. Examples include trace.from_Tomcat01.duration_mean (~0.0806), many trace.from_MG02.* and trace.from_Tomcat03/* rows, and trace.duration_p95 entries.
- No requested keys were missing — all target components/patterns were present but some pattern matches returned no rows outside the returned sets (see returned lists). The missing_notes list was empty.

Notable timing relationship observed
- The trace rows in this extraction are at timestamp 1615331040 (≈ 07:04:00+08), and many metric rows are at timestamp 1615331100 (≈ 07:05:00+08). This matches the earlier correlation pattern: brief upstream trace latency events occur ~1 minute before the DB/host metric rows in the window you requested (suggesting upstream latency may precede metric spikes).

Quick interpretation & suggested next steps
- The metric rows show InnoDB activity at these minutes (large "data written", dblwr activity, fsyncs, etc.). The trace rows show inter-service latency/row-counts on MG/docker/Tomcat routes in the minute immediately before the Mysql01 metrics.
- To investigate further, you can:
  - Inspect the metric rows at timestamps 07:05–07:06 specifically (we saw earlier InnoDB row-lock spikes at 07:06 in the anomalies); I can return metric rows for 07:05–07:07 (or show metric_anomalies rows) if you want.
  - Request more detailed trace rows around the specific trace timestamps (e.g., 07:04–07:06) or show the full anomaly rows for trace_anomalies/metric_anomalies for side-by-side inspection.

If you want, I can display selected representative rows (e.g., the first 10 of each returned DataFrame) or expand the window slightly to capture the 07:06 InnoDB row-lock spike. Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp  cmdb_id                                       kpi_name       value
0   1615331100  Mysql01          Mysql-MySQL_3306_Innodb Row Lock Time      0.0000
1   1615331100  Mysql01  Mysql-MySQL_3306_Innodb os log pending writes      0.0000
2   1615331100  Mysql01         Mysql-MySQL_3306_Innodb os log written    742.4000
3   1615331100  Mysql01          Mysql-MySQL_3306_Innodb pages created      0.0000
4   1615331100  Mysql01             Mysql-MySQL_3306_Innodb pages read      0.0000
..         ...      ...                                            ...         ...
45  1615331100  Mysql01            Mysql-MySQL_3306_Innodb data fsyncs      2.4833
46  1615331100  Mysql01     Mysql-MySQL_3306_Innodb log write requests      0.3833
47  1615331100  Mysql01           Mysql-MySQL_3306_Innodb dblwr writes      0.6500
48  1615331100  Mysql01    Mysql-MySQL_3306_Innodb dblwr pages written      2.4000
49  1615331100  Mysql01           Mysql-MySQL_3306_Innodb data written  79539.2000

[50 rows x 4 columns],      timestamp   cmdb_id                         trace_name      value
0   1615331040      MG01  trace.from_Tomcat01.duration_mean   0.080625
1   1615331040  dockerB1          trace.from_MG02.row_count  19.000000
2   1615331040  dockerB1       trace.from_MG02.duration_p95   0.135200
3   1615331040  dockerB1      trace.from_MG02.duration_mean   0.048842
4   1615331040  dockerB1          trace.from_MG01.row_count  28.000000
..         ...       ...                                ...        ...
45  1615331040      MG01      trace.from_Tomcat03.row_count  29.000000
46  1615331040      MG01   trace.from_Tomcat03.duration_p95   0.115800
47  1615331040      MG01  trace.from_Tomcat03.duration_mean   0.059759
48  1615331040      MG01      trace.from_Tomcat02.row_count  40.000000
49  1615331040      MG01   trace.from_Tomcat02.duration_p95   0.560500

[50 rows x 4 columns], [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# Reuse existing variables from kernel if present: log_df, tz
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp -> timezone-aware datetime column 'ts'
if 'ts' not in log_df.columns:
    log_df['ts'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    log_df['ts'] = pd.to_datetime(log_df['ts']).dt.tz_convert(tz)

# Define target components and window (UTC+8)
components = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04','apache01','apache02']
window_start = pd.Timestamp('2021-03-10 07:04:00').tz_localize(tz)
window_end   = pd.Timestamp('2021-03-10 07:08:00').tz_localize(tz)

# Filter logs by time window and components
mask_time = (log_df['ts'] >= window_start) & (log_df['ts'] <= window_end)
mask_comp = log_df['cmdb_id'].isin(components)
logs_window_all = log_df[mask_time & mask_comp].copy()

# Select requested columns and sort by timestamp ascending, limit to 50 rows
logs_out = logs_window_all.sort_values('ts').loc[:, ['timestamp','cmdb_id','log_name','value']].head(50).reset_index(drop=True)

# Aggregate per requested component: total rows and count of log.error_count with value>0
agg_all = logs_window_all.groupby('cmdb_id').agg(
    rows_in_window=('timestamp','count'),
    error_count_positive=('value', lambda v: ((logs_window_all.loc[v.index, 'log_name']=='log.error_count') & (v>0)).sum())
).reset_index().rename(columns={'cmdb_id':'component'})

# Ensure all target components present in aggregation (fill zeros for missing)
agg_df = pd.DataFrame({'component': components}).merge(agg_all, on='component', how='left').fillna({'rows_in_window':0,'error_count_positive':0})
agg_df['rows_in_window'] = agg_df['rows_in_window'].astype(int)
agg_df['error_count_positive'] = agg_df['error_count_positive'].astype(int)

# Notes about missing components or missing log_name occurrences
missing_notes = []
missing_components = agg_df[agg_df['rows_in_window']==0]['component'].tolist()
if missing_components:
    missing_notes.append(f"Components with no rows in window: {missing_components}")

# Check presence of 'log.error_count' overall and per component
error_name_present = logs_window_all['log_name'].eq('log.error_count').any()
if not error_name_present:
    missing_notes.append("No 'log.error_count' entries found in this window for the selected components.")
else:
    comps_with_error = logs_window_all[logs_window_all['log_name']=='log.error_count']['cmdb_id'].unique().tolist()
    comps_without_error = [c for c in components if c not in comps_with_error]
    if comps_without_error:
        missing_notes.append(f"Components with no 'log.error_count' entries in window: {comps_without_error}")

# Final compact outputs: logs_out (<=50 rows), agg_df (per-component summary), missing_notes
logs_out, agg_df, missing_notes

```
Out[8]:
```
summary = (
    "Log extraction summary (window 2021-03-10 07:04:00 to 2021-03-10 07:08:00 UTC+8):\n\n"
    f"- Rows returned: {len(logs_out)} (limited to 50), sorted by timestamp ascending.\n"
    "- Components inspected: Tomcat01, Tomcat02, Tomcat03, Tomcat04, apache01, apache02.\n\n"
    "- Per-component counts in the window:\n"
    f"  - Tomcat01: {int(agg_df.loc[agg_df.component=='Tomcat01','rows_in_window'].iloc[0])} rows\n"
    f"  - Tomcat02: {int(agg_df.loc[agg_df.component=='Tomcat02','rows_in_window'].iloc[0])} rows\n"
    f"  - Tomcat03: {int(agg_df.loc[agg_df.component=='Tomcat03','rows_in_window'].iloc[0])} rows\n"
    f"  - Tomcat04: {int(agg_df.loc[agg_df.component=='Tomcat04','rows_in_window'].iloc[0])} rows\n"
    f"  - apache01: {int(agg_df.loc[agg_df.component=='apache01','rows_in_window'].iloc[0])} rows\n"
    f"  - apache02: {int(agg_df.loc[agg_df.component=='apache02','rows_in_window'].iloc[0])} rows\n\n"
    "- Error counts (log_name == 'log.error_count' and value > 0) within the window:\n"
    "  - All listed components show 0 positive error_count rows in this window.\n\n"
    "- Notes:\n"
    "  - No requested components or log_name types were missing from the data for this window.\n"
    "  - If you want, I can show the first 10 log rows returned or expand the time window to capture nearby events."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp   cmdb_id         log_name  value
0   1615331040  Tomcat01  log.error_count    0.0
1   1615331040  apache02    log.row_count  124.0
2   1615331040  apache02  log.error_count    0.0
3   1615331040  apache01  log.error_count    0.0
4   1615331040  Tomcat04    log.row_count   59.0
..         ...       ...              ...    ...
45  1615331220  Tomcat02    log.row_count   54.0
46  1615331220  Tomcat02  log.error_count    0.0
47  1615331220  Tomcat01    log.row_count   48.0
48  1615331220  Tomcat01  log.error_count    0.0
49  1615331220  Tomcat03    log.row_count   37.0

[50 rows x 4 columns],   component  rows_in_window  error_count_positive
0  Tomcat01              10                     0
1  Tomcat02              11                     0
2  Tomcat03              11                     0
3  Tomcat04              11                     0
4  apache01              10                     0
5  apache02              10                     0, [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```